# 03: Conformance assessment

Ports `build_conformance_voltvar.py` and `build_conformance_voltwatt.py` to DuckDB over the local store. 

Every AS/NZS 4777.2 expression is imported from `bms_sa_review.shared.as4777_curves` and runs unchanged

Three questions are kept separate:
1. **Maximum-output / required-Q conformance**: Did the inverter do what the curve asks?
2. **Whether the observation was informative** about an active response.
3. **Counterfactual-supported curtailed energy**: *not available yet*, needs D12.

The 10% site threshold is a project reporting convention, not a tolerance granted by AS/NZS 4777.2.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_contract as contract, se_params
from solar_edge.lib import se_conformance as cf
from solar_edge.lib import se_adverse as adv
from solar_edge.lib import se_plots as plots
from solar_edge.lib import se_spatial as spat

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect()
config = se_params.CONFIG
params = se_params.PARAMS
display(contract.manifest(config, params).query("section != 'convention'"))

## 1. Volt-VAr conformance

Interval scoring follows the CTE chain of `build_conformance_voltvar.build_sql`:
required Q → ±4% tolerance band → AS4777 Figure 2.1 capability clamp → `Q_impact` → five categories.

Two assumptions:

- **Capacity basis is `s_99`**, not nameplate. It scales the required-Q curve, the tolerance band, the 20% assessability rule and the capability floor. Because `s_99` is an *observed* p99, a site that never approached its inverter limit gets a low `s_99`, and a smaller required Q
- **`capability_profile="review_corrected"`**: Reactive-power priority above 0.8·S, and intervals below 0.2·S marked unassessable rather than scored, since Figure 2.1 sets no quantified minimum there.

BMS update: `reduced_nonconf` = adverse + inactive + significant shortfall. `Q_near_conformant` is excluded: those inverters deliver 90–110% of required reactive power.

In [ ]:
vvar_site_day = cf.voltvar_site_day(con, config, params)
print(f"{len(vvar_site_day):,} site-days scored")

vvar_summary = cf.voltvar_summary(vvar_site_day, by_cohort=True)
display(vvar_summary.T)

In [ ]:
display(plots.plot_q_categories(vvar_summary))

### Same result unpacked:

| Measure | Denominator | Answers |
|---|---|---|
| **% of assessable intervals** | every interval counts once | how *often* it happens |
| **% of sites** | every site counts once, 10% rule per category | how *many inverters* |
| **kVArh** | none. Absolute total | how much reactive support was *actually missing* |
| **kVArh / kW / h** | site rating × assessable hours | how *badly*, per inverter, per hour |

In [ ]:
vvar_measures = cf.voltvar_measures(vvar_site_day, config)
display(plots.plot_conformance_dashboard(vvar_measures))
display(vvar_measures)

## 2. Volt-VAr site verdicts

**The 10% rule, exactly as CICCADA milestone 3 applies it** (`conformance_metrics.aggregate_sites`):

```
nonconf_frac = reduced_nonconf_count / capability_assessable_count
conformant   = nonconf_frac <  0.10      # STRICTLY less than
nonconformant= nonconf_frac >= 0.10
```

Obs:

- The denominator is **capability-assessable intervals**, not all intervals and not
  voltage-exposed intervals. Below 0.2·S the standard sets no quantified requirement.
- The comparison is **strict**. A site sitting exactly on 0.10 is non-conformant.

In [ ]:
vvar_verdicts = cf.voltvar_site_verdicts(vvar_site_day, config)
display(pd.crosstab(
    vvar_verdicts.is_three_phase.map({False: "single-phase", True: "three-phase"}),
    vvar_verdicts.verdict, normalize="index").round(4) * 100)
display(pd.crosstab(
    vvar_verdicts.is_three_phase.map({False: "single-phase", True: "three-phase"}),
    vvar_verdicts.verdict, margins=True))

### Aggregate verdict

In [ ]:
vvar_verdict_measures = cf.site_verdict_measures(vvar_site_day, config)
display(plots.plot_site_verdicts(vvar_verdict_measures, config))
display(vvar_verdict_measures)

#### Denominator conventions: Volt-VAr

**Which sites belong in the denominator?** Two defensible answers, and they are *not* the difference between "conformant" and "not tested".

| convention | denominator | question |
|---|---|---|
| `assessed` | sites with ≥ 1 interval in the assessment population | *of the inverters actually tested, how many passed?* |
| `all` | every site in the cohort | *what share of the fleet has demonstrated conformance?* |

`assessed` is what the original study did. 

In `Volt-Watt-Trino.ipynb` the build carries `HAVING avg(voltage) > 253`, so a site that never saw high voltage produces **no rows at
all** and cannot reach any downstream rate; `bms_sa_review.conformance_metrics.aggregate_sites`
does the same explicitly with `d[d[denominator] >= min_intervals]`, and `fleet_result`
keeps `n_sites_contained_in_table` and `n_assessed_sites` as separate fields for exactly
this reason.

**Neither convention counts a never-tested site as conformant.** 

In [ ]:
vvar_denominators = cf.denominator_comparison(vvar_site_day, "voltvar", config)
display(vvar_denominators.drop(columns="denominator_intervals"))
cf.denominator_note(vvar_denominators)

## 3. Volt-VAr breakdowns

Two views per dimension, deliberately kept apart because they have different
denominators and answer different questions:

- **`site_pct`**: % of *sites* conformant / non-conformant, on the 10% rule. Equal weight per site.
- **`interval_pct`**: % of capability-assessable *intervals* in each category. Weighted by how much each site was observed.

In [ ]:
for dimension, label in [("site_state", "STATE"),
                         ("capacity_band", "SYSTEM SIZE (s_99, kVA)"),
                         ("cohort", "PHASE COHORT")]:
    site_pct, interval_pct = cf.voltvar_breakdown(con, vvar_site_day, by=dimension,
                                                  config=config)
    print(f"\n{'=' * 100}\n{label} — % of SITES by verdict (10% rule)\n{'=' * 100}")
    display(site_pct)
    print(f"{label} — % of ASSESSABLE INTERVALS by category")
    display(interval_pct)
    display(plots.plot_breakdown_rates(
        site_pct, interval_pct, dimension, "pct_reduced_nonconf",
        label=f"Volt-VAr — {label}"))

### By postcode

In [ ]:
pc_sites, pc_intervals = cf.voltvar_breakdown(con, vvar_site_day, by="postcode",
                                              config=config)
big = pc_intervals[pc_intervals.n_sites >= 5].copy()
print(f"{len(pc_intervals):,} postcodes; {len(big):,} with at least 5 sites\n")
print("Highest reduced non-conformance:")
display(big.nlargest(10, "pct_reduced_nonconf"))
print("Lowest:")
display(big.nsmallest(10, "pct_reduced_nonconf"))

pc_intervals.to_csv(C.ARTEFACT_DIR / "voltvar_by_postcode.csv", index=False)
pc_sites.to_csv(C.ARTEFACT_DIR / "voltvar_sites_by_postcode.csv", index=False)

#### Mapped

A **bubble map, not a choropleth** — and that is a methodological choice, not a
convenience.

Australian postcode areas span more than four orders of magnitude: 4702 covers roughly
50,000 km², an inner-Sydney postcode a couple. Filling polygons by rate hands nearly the
whole visual field to a few enormous rural postcodes holding a handful of sites, while the
dense metropolitan postcodes where most of the fleet actually lives shrink to specks. The
eye then weights the map by land area — the one variable here that carries no information.

So marker **area = number of sites**, **colour = rate**, outlines faint underneath for
context. Postcodes below the 5-site threshold are drawn hollow rather than deleted: a 100%
rate over two sites is not a finding, but hiding it would misrepresent where the fleet is
thin.

In [ ]:
fig = plots.plot_postcode_map(
    con, pc_intervals, "pct_reduced_nonconf", min_sites=5,
    label="Volt-VAr reduced non-conformance (% of assessable intervals)")
display(fig)

The capitals are overlapping clusters at continental scale, so the same data zoomed.

In [ ]:
METRO = {
    "Adelaide":              (138.35, 139.05, -35.35, -34.55),
    "South-east Queensland": (152.55, 153.65, -28.35, -26.95),
    "Sydney / Illawarra":    (150.45, 151.55, -34.65, -33.45),
}

# Colour limits come from the national frame, not each panel, so the three zooms
# are comparable with each other and with the map above.
for name, extent in METRO.items():
    display(plots.plot_postcode_map(
        con, pc_intervals, "pct_reduced_nonconf", min_sites=5,
        extent=extent, figsize=(8.0, 7.0),
        label="Reduced non-conformance (% of assessable intervals)",
        title=f"{name} — Volt-VAr"))

#### Is postcode actually explanatory?

The map invites the conclusion that geography drives non-conformance. Before accepting
that, the claim needs testing — and the obvious test is the wrong one.

Regressing the site rate on postcode gives R² ≈ 0.46, which looks decisive. It is an
artefact. **194 of 507 postcodes hold exactly one site**; the median is 2. Where a postcode
has one site, "the postcode effect" and "that site's behaviour" are the same number, and a
507-level categorical fitted to ~1,600 observations has enough free parameters to memorise
the data. Shuffling the postcode labels at random still yields R² ≈ 0.32 — that is the
price of the parameters alone, before any signal.

So every measure below is compared against a **permutation null** that preserves the same
group-size structure and the same capacity to overfit. Three tests, because they answer
different questions and can disagree:

1. **ICC** — are sites in the same postcode alike? A *label* question.
2. **Moran's I** — are *nearby* postcodes alike? A *spatial* question.
3. **Nested R²** — does postcode add anything once state, size and cohort are known?

In [ ]:
effect = spat.postcode_effect_report(con, vvar_site_day, config)

**Reading the result.** The variation is spatially structured and highly significant, but
the scale matters: ~500 postcode parameters buy barely more than 3 state ones
(r²_excess 0.137 vs 0.127). The gradient is **regional, not per-postcode** — consistent
with the static maps, where Adelaide and Sydney run hot while south-east Queensland runs
cool.

Practical consequence: report conformance by **state or metro area**. Per-postcode rates
mostly resolve sampling noise, and the map above should be read as a way of *locating*
sites to inspect, not as a per-postcode measurement.

And none of this is causal. Postcode is a stand-in for network topology, feeder length,
transformer sizing, installer and inverter vintage — none of which are in this dataset.
A postcode effect says "look at the network here", not "the postcode caused it".

#### Interactive map

Rate as colour, **installs as circle area**, both in the tooltip — so a lurid red circle
backed by two sites is immediately identifiable as noise rather than a finding, which is
exactly the confusion the static map cannot prevent.

Needs `folium` (`conda install -c conda-forge folium`). Saved to `artefacts/` as
standalone HTML you can send to someone without the repo.

In [ ]:
imap = spat.interactive_postcode_map(
    con, pc_intervals, "pct_reduced_nonconf",
    min_sites=1,                       # show everything; size encodes the caveat
    label="Volt-VAr reduced non-conformance",
    out_path=C.ARTEFACT_DIR / "voltvar_postcode_map.html",
)
# imap

## 4. Unpacking the adverse results

`Q_adverse` means the inverter moved reactive power in the **wrong direction**: Supplying where the curve requires absorption.

The triage uses **magnitude**, which is orientation-independent:

| class | meaning | safe to report as non-conformance? |
|---|---|---|
| `polarity_suspect` | adverse in direction, but \|Q\| tracks the required curve within ±4% | **No**: Might be a data-format finding |
| `genuinely_adverse` | adverse in direction, magnitude does not match the curve | **Yes**: adverse under either orientation |
| `adverse_but_inactive` | adverse in direction, delivering < 25% of requirement | **No**: direction of a near-zero quantity |
| `not_adverse` | absorbing, or near zero | — |

This is **triage, not a correction.** Flipping the polarity-suspect sites and re-scoring
would infer the sign from conformity with the curve and then report conformity with the
curve.

In [ ]:
adverse = adv.classify_adverse_sites(con, config)
display(pd.crosstab(adverse.adverse_class, adverse.cohort, margins=True))
display(adv.adverse_summary(adverse))

### Where the adverse intervals actually come from

Site counts and interval counts tell different stories. This attributes every
`Q_adverse` interval to its site's class, which turns "the adverse rate may be
contaminated" into a number.

In [ ]:
impact_tbl = adv.adverse_conformance_impact(con, adverse, config, params)
impact_tbl["pct_of_all_adverse"] = (
    100 * impact_tbl.adverse_intervals / impact_tbl.adverse_intervals.sum()).round(2)
display(impact_tbl)

adverse.to_csv(C.ARTEFACT_DIR / "adverse_classification.csv", index=False)
print(f"Per-site classification -> {C.ARTEFACT_DIR / 'adverse_classification.csv'}")

### Per-site table

One row per site with its category percentages, so you can pick sites to inspect rather
than guess. Percentages are against **that site's own** capability-assessable intervals,
which puts a 200-interval site and a 200,000-interval site on the same scale — so
`assessable_intervals` is carried alongside, because a 100% rate on a handful of
intervals is not a finding.

`adverse_class` is joined from the triage in section 4, so polarity-suspect sites can be inspected first or set aside.

Take any `site_alias` from this table into `05_site_explorer.ipynb`:

```python
from solar_edge.lib import se_explore as ex
ex.site_profile(con, "AUS978")
ex.plot_site_voltvar_curve(ex.site_voltvar_curve(con, "AUS978"), "AUS978")
```

In [ ]:
vvar_sites = cf.voltvar_site_table(con, vvar_site_day, config, adverse=adverse)
display(vvar_sites.verdict.value_counts())

print("\nWorst 15 by reduced non-conformance:")
display(vvar_sites.head(15))

print("Best 15 (conformant, most-observed first):")
display(vvar_sites[vvar_sites.verdict == "conformant"]
        .nlargest(15, "assessable_intervals"))

vvar_sites.to_csv(C.ARTEFACT_DIR / "voltvar_by_site.csv", index=False)
print(f"\n-> {C.ARTEFACT_DIR / 'voltvar_by_site.csv'}  ({len(vvar_sites):,} sites)")

# The Volt-Watt site table needs vwatt_site_day, which is built in section 5.
# It is written there rather than here.

## 5. Volt-Watt conformance

### 5a. Volt-Watt maximum-output conformance:

> **Question:** when voltage selected a reduced maximum active-power level, did measured P exceed that permitted maximum (including the ±4% band)?

A site is *exposed* above 253 V, and non-conformant when measured P exceeds the Volt-Watt ceiling plus tolerance.
A below-ceiling observation satisfies this maximum-output test but does **not**, by itself, demonstrate active curtailment.

### 5b. Volt-Watt response-supported result:

> **Question:** was enough solar power available to test the response, or was a violation directly observed?

Port of `conformance_voltwattghi` from `Volt-Watt-ghi.ipynb`. An exposed interval counts in the denominator when

```
uncurtailed_P > max_P_volt_watt   OR   uncurtailed_P IS NULL
```

Two outputs:

| column | meaning |
|---|---|
| `nonconformance` | kW generated **above** the ceiling — a violation |
| `curtailment` | kW available but not generated while staying below the ceiling — **evidence of correct response**, not a fault |

Requires `se_uncurtailedpv`.

In [ ]:
vwatt_site_day = cf.voltwatt_site_day(con, config)
vwatt_summary = cf.voltwatt_summary(vwatt_site_day, by_cohort=True)
display(vwatt_summary.T)

vwatt_verdicts = cf.voltwatt_site_verdicts(vwatt_site_day, config)
display(pd.crosstab(
    vwatt_verdicts.is_three_phase.map({False: "single-phase", True: "three-phase"}),
    vwatt_verdicts.verdict, margins=True))

### Volt-Watt site verdicts

- the denominator is **exposed** intervals (V > 253 V). 
- Volt-Watt has no 20%-of-rating floor (like voltvar), but a site that never saw high voltage was never tested.
- hence the third verdict, `not exposed`, which is **not** a pass.
- the energy is **kWh generated above the ceiling**, not kVArh of missing reactive power.
- Volt-Watt non-conformance means producing too much, so the quantity is real energy that should not have been generated (exported?).

In [ ]:
# `denominator="all"` keeps never-exposed sites visible as their own category.
vwatt_verdict_measures = cf.voltwatt_verdict_measures(
    vwatt_site_day, config, denominator="all")
display(plots.plot_site_verdicts(vwatt_verdict_measures, config,
                                 measures=cf.VW_VERDICT_MEASURES, mode="Volt-Watt",
                                 denominator="all sites in the cohort"))
display(vwatt_verdict_measures)

#### Denominator conventions: Volt-Watt

**Which sites belong in the denominator?** 

| convention | denominator | question |
|---|---|---|
| `assessed` | sites with ≥ 1 interval in the assessment population | *of the inverters actually tested, how many passed?* |
| `all` | every site in the cohort | *what share of the fleet has demonstrated conformance?* |

- `assessed` is what's in the Milestone report #3 
- In (`Volt-Watt-Trino.ipynb`) the build carries `HAVING avg(voltage) > 253`, so a site that never saw high voltage produces **no rows at all** and cannot reach any downstream rate

In [ ]:
vwatt_denominators = cf.denominator_comparison(vwatt_site_day, "voltwatt", config)
display(vwatt_denominators.drop(columns="denominator_intervals"))
cf.denominator_note(vwatt_denominators)

In [ ]:
vwatt_measures_assessed = cf.voltwatt_verdict_measures(
    vwatt_site_day, config, denominator="assessed")
display(plots.plot_site_verdicts(
    vwatt_measures_assessed, config, measures=cf.VW_VERDICT_MEASURES,
    mode="Volt-Watt", denominator="tested sites only — Solar Analytics convention"))
display(vwatt_measures_assessed)

In [ ]:
vwatt_sites = cf.voltwatt_site_table(con, vwatt_site_day, config)
display(vwatt_sites.head(10))

vwatt_sites.to_csv(C.ARTEFACT_DIR / "voltwatt_by_site.csv", index=False)
print(f"-> {C.ARTEFACT_DIR / 'voltwatt_by_site.csv'} ({len(vwatt_sites):,} sites)")

### Volt-Watt breakdowns

- Same two-frame structure as Volt-VAr. Note the interval denominator here is **exposed** intervals (V > 253 V).
- `severity_Wh_per_kVA_per_exposed` is the normalised magnitude metric: nonconformance Wh ÷ (capacity × exposed intervals). 
- Frequency says how *often*; severity says how far *over*. It does not replace the 10% classification.

In [ ]:
for dimension, label in [("site_state", "STATE"),
                         ("capacity_band", "SYSTEM SIZE (s_99, kVA)"),
                         ("cohort", "PHASE COHORT")]:
    site_pct, interval_pct = cf.voltwatt_breakdown(con, vwatt_site_day, by=dimension,
                                                   config=config)
    print(f"\n{'=' * 100}\n{label} — % of SITES by verdict (10% rule)\n{'=' * 100}")
    display(site_pct)
    print(f"{label} — exposure, non-conformance and severity")
    display(interval_pct)
    display(plots.plot_breakdown_rates(
        site_pct, interval_pct, dimension, "pct_intervals_nonconformant",
        label=f"Volt-Watt — {label}"))

In [ ]:
pc_sites_vw, pc_intervals_vw = cf.voltwatt_breakdown(con, vwatt_site_day, by="postcode",
                                                     config=config)
big_vw = pc_intervals_vw[pc_intervals_vw.sites_exposed >= 5]
print(f"{len(pc_intervals_vw):,} postcodes; {len(big_vw):,} with >= 5 exposed sites\n")
display(big_vw.nlargest(10, "pct_intervals_nonconformant"))
pc_intervals_vw.to_csv(C.ARTEFACT_DIR / "voltwatt_by_postcode.csv", index=False)

### 5b: Response supported

In [ ]:
try:
    vwghi_site_day = cf.voltwatt_ghi_site_day(con, config)
    vwghi_summary = cf.voltwatt_ghi_summary(vwghi_site_day, by_cohort=True)
    display(vwghi_summary.T)

    # voltwatt_verdict_measures reads total_count / nonconformance_* , which the
    # response-supported frame carries under the same names — so it applies
    # unchanged, just with a response-supported denominator.
    vwghi_measures = cf.voltwatt_verdict_measures(vwghi_site_day, config)
    display(plots.plot_site_verdicts(
        vwghi_measures, config, measures=cf.VW_VERDICT_MEASURES,
        mode="Volt-Watt (response-supported)",
        denominator="P above the ceiling, where power was available"))
    display(vwghi_measures)

    vwghi_site_day.to_parquet(C.STORE_DIR / "se_conformance_voltwattghi.parquet",
                              index=False)
except FileNotFoundError as exc:
    vwghi_site_day = None
    print("5b not runnable yet:\n"); print(exc)

#### 5a vs 5b

The comparison only means something if `pct_supported_by_model` is large. Where it is
small, 5b's denominator is mostly the `uncurtailed_P IS NULL` fallback and the two tables
are measuring the same thing twice.

`demonstrated_curtailment_intervals` is the row 5a cannot produce at all: intervals where
the counterfactual proves power was available and the inverter held below the ceiling
anyway. That is Volt-Watt working, observed rather than assumed.

In [ ]:
if vwghi_site_day is not None:
    basic = cf.voltwatt_summary(vwatt_site_day, by_cohort=True)

    # Built column by column rather than by renaming: voltwatt_ghi_summary already
    # HAS an `exposed_intervals`, so renaming `response_supported_intervals` onto
    # that name produces a duplicate column and concat refuses to align it.
    # Naming the denominator explicitly is also the point of the table — the two
    # rates are not measured against the same population.
    comparison = pd.concat([
        pd.DataFrame({
            "variant": "5a basic (max-output)",
            "cohort": basic.cohort,
            "denominator": "exposed (V > 253 V)",
            "denominator_intervals": basic.exposed_intervals,
            "nonconformant_intervals": basic.nonconformant_intervals,
            "pct_nonconformant": basic.nonconformant_pct_of_exposed,
        }),
        pd.DataFrame({
            "variant": "5b response-supported",
            "cohort": vwghi_summary.cohort,
            "denominator": "response-supported",
            "denominator_intervals": vwghi_summary.response_supported_intervals,
            "nonconformant_intervals": vwghi_summary.nonconformant_intervals,
            "pct_nonconformant": vwghi_summary.pct_nonconformant_of_supported,
        }),
    ], ignore_index=True)
    display(comparison.sort_values(["cohort", "variant"]).reset_index(drop=True))

    print("Model-backed share of the 5b denominator — read this before quoting 5b:")
    display(vwghi_summary[["cohort", "exposed_intervals", "pct_covered",
                           "pct_supported_by_model",
                           "demonstrated_curtailment_intervals",
                           "demonstrated_curtailment_kWh"]])

    weak = vwghi_summary[vwghi_summary.pct_supported_by_model < 5]
    if len(weak):
        print(f"\nWARNING: {', '.join(weak.cohort)} — under 5% of the 5b denominator is")
        print("model-backed. The rest reached it through the `uncurtailed_P IS NULL`")
        print("fallback, which applies the 5a test. Any 5a/5b difference there is noise,")
        print("not evidence. Fix coverage before reporting 5b separately.")

## 6. Population funnel

The gap between *exposed* and *assessable* is the point. A site can sit in the Volt-VAr
band all year and never be assessable, because below 20% of rated power Figure 2.1 sets
no quantified minimum capability. A conformance rate quoted without this table invites
the reader to assume the denominator is the whole fleet.

In [ ]:
display(cf.conformance_funnel(vvar_site_day, vwatt_site_day))

### Derating-flag corroboration

Volt-Watt exposure is the one place where the inverter-reported `derating_active` flag
and the standard's own ceiling should agree. A high share of exposed intervals carrying
the flag means it is at least partly grid-voltage-driven — the premise Method C rests on
at D14.

In [ ]:
exposed = vwatt_summary.exposed_intervals.sum()
flagged = vwatt_summary.exposed_with_derating_flag.sum()
print(f"Exposed intervals (V > 253 V):     {exposed:,}")
print(f"  with derating_active set:        {flagged:,}  ({100 * flagged / exposed:.1f}%)")
print()
print("Recall from D3 that the raw flag is 1.0 or NULL, never 0.0, so precision against")
print("it is interpretable but recall is not. D14 states that limitation.")

## 7. Persist

Site-day grain, matching `conformance_voltvar_v2` / `conformance_voltwatt_v2` so the two
studies compare directly. Written to the store, not the repository.

In [ ]:
for name, frame in (("se_conformance_voltvar", vvar_site_day),
                    ("se_conformance_voltwatt", vwatt_site_day)):
    path = C.STORE_DIR / f"{name}.parquet"
    frame.to_parquet(path, index=False)
    print(f"{name}: {len(frame):,} rows -> {path}")

## 8. Both responses together

### The population problem, first

A plain 2×2 of the two verdicts would be misleading, because **the two modes are not
asked of the same sites**. Every site here has capability-assessable Volt-VAr intervals,
but only ~29% ever exceed 253 V. So "conforms to Volt-VAr only" would overwhelmingly mean
*"was never tested on Volt-Watt"* — a statement about voltage exposure, not about the
inverter.

`joint_class` is therefore defined **only over sites testable on both**. Everything else is
labelled `Volt-Watt not tested` / `Volt-VAr not assessable` and held out of the four-way
comparison rather than quietly counted as conforming.

The full cross-tab is shown first anyway, so the exclusion is visible rather than asserted.

In [ ]:
joint = cf.joint_site_verdicts(con, vvar_site_day, vwatt_site_day, config)

print("FULL cross-tab — every site, including those never tested on Volt-Watt:")
display(pd.crosstab(joint.vvar_verdict, joint.vwatt_verdict, margins=True))

joint_summary = cf.joint_conformance_summary(joint)
display(joint_summary)

### Breakdowns

100% stacked, because the question is the **mix** within each group, not the group's size —
absolute counts would just redraw the fleet's size distribution. `n` is on every bar, so a
100%-of-four bar cannot be mistaken for a finding.

Colour carries the reading: green passes both, blue/orange pass one, red passes neither. A
group that is mostly one of the middle colours has a **single-mode** problem — one response
configured, the other not — which is a different remedy from an inverter failing both.

In [ ]:
for dimension, label in [("cohort", "phase cohort"),
                         ("capacity_band", "system size (s_99, kVA)"),
                         ("state", "state")]:
    bd = cf.joint_breakdown(joint, dimension)
    print(f"\n{'=' * 90}\nJOINT CONFORMANCE by {label.upper()}\n{'=' * 90}")
    display(bd)
    display(plots.plot_joint_conformance(
        bd, dimension, label=f"Joint Volt-VAr / Volt-Watt conformance by {label}"))

### Are the two failures independent?

This is what the cross-tab is really for. If the two failures co-occur more than chance,
the likely story is **per-inverter** — a commissioning or firmware state that disables both
responses together. If they are independent, they are two separate settings that happen to
be wrong in different populations, and the fixes differ.

Reported as an odds ratio with a **permutation** p-value (shuffling one verdict against the
other), which needs no distributional assumption and copes with the small cell counts this
2×2 usually has. The Haldane correction keeps it finite when a cell is zero — which also
makes it look sturdier than it is, so `smallest_cell` is reported and the reading is
downgraded to *descriptive only* below 10.

In [ ]:
independence = cf.joint_independence(joint)
for k, v in independence.items():
    print(f"  {k:18} {v}")

### The sites that conform to only one

The handoff to notebook 05. `Volt-VAr only` and `Volt-Watt only` are the interesting
populations: something is configured correctly on one response and not the other, in the
same inverter — which is much harder to explain away as a data artefact than a site
failing everything.

In [ ]:
one_only = joint[joint.joint_class.isin(["Volt-VAr only", "Volt-Watt only"])].copy()
display(one_only.joint_class.value_counts())

cols = ["site_alias", "joint_class", "cohort", "state", "postcode", "s_99",
        "capacity_band", "vvar_verdict", "vvar_nonconf_frac", "vvar_assessable",
        "vwatt_verdict", "vwatt_nonconf_frac", "vwatt_exposed"]
display(one_only[cols].sort_values(["joint_class", "vvar_nonconf_frac"]).head(20))

joint.to_csv(C.ARTEFACT_DIR / "joint_conformance_by_site.csv", index=False)
print(f"\nAll {len(joint):,} sites -> "
      f"{C.ARTEFACT_DIR / 'joint_conformance_by_site.csv'}")